# Manga semantic search lab

Thin Colab/Jupyter frontend for `semantic_search_lab.py`. It indexes ZIP archives or manga directories, then opens a standalone comparison report.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'devscripts/semantic_search_lab.py').exists()), None)
if REPO is None:
    REPO = Path('/content/manga-image-translator')
    if not REPO.exists():
        subprocess.run(['git', 'clone', 'https://github.com/huynguyen1999/manga-translator.git', str(REPO)], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers', 'Pillow', 'numpy==1.26.4'], check=True)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available() else 'cpu')
print(f'Using {DEVICE}')

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    INPUTS = list(uploaded)
except ImportError:
    INPUTS = ['/path/to/manga.zip']  # Replace with one or more local ZIPs or manga directories

INDEX = REPO / 'devscripts/data/semantic_search'
subprocess.run([sys.executable, 'devscripts/semantic_search_lab.py', 'index', '--input', *INPUTS, '--output', str(INDEX), '--device', DEVICE], check=True)

In [ ]:
QUERY = 'a girl learns that someone close to her betrayed her'
REPORT = INDEX / 'reports' / 'comparison.html'
subprocess.run([sys.executable, 'devscripts/semantic_search_lab.py', 'compare', '--index', str(INDEX), '--query', QUERY, '--top-k', '10', '--report', str(REPORT), '--device', DEVICE], check=True)
from IPython.display import IFrame, display
display(IFrame(REPORT.as_posix(), width='100%', height=900))